<a href="https://colab.research.google.com/github/raffaellaalpaca-ai/ds2002-fa26/blob/main/2026_09_23_%E2%80%94_Cleaning_Clinic_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [3]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [4]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [5]:
# TODO
print("Shape: ", df.shape)
print("Dtypes: ", df.dtypes)
print("Null count per column: ", df.isnull().sum())
print("Number of exact duplicate rows: ", df.duplicated().sum())
print(df)

Shape:  (8, 6)
Dtypes:  order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object
Null count per column:  order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
Number of exact duplicate rows:  1
   order_id           item   category  qty   price                   ts
0         1   Cheeseburger       Food  2.0   $7.50  2026-09-05T12:03:00
1         1   Cheeseburger       Food  2.0   $7.50  2026-09-05T12:03:00
2         2  cheese burger       food  1.0     7.5     09/05/2026 12:40
3         3    Foam Finger      Merch  NaN      12  2026-09-05 13:00:00
4         4   UVA T-Shirt     Apparel  2.0  $24.00     2026-09-05 13:05
5         5    Rain Poncho   RainGear -3.0       6  2026-09-05T13:20:00
6         6    rain poncho  rain-gear  4.0   $6.00                  NaN
7         7            NaN      Merch  1.0      12  2026-09-05T14:00:00


**What is wrong with this data?** List at least five specific problems:

1. There is one exact duplicate row for order 1
2. There are missing values in the item, qty, and ts columns.
3. The price column has inconsistent format, such as $7.50, 7.5, $24.00, and 12.
4. The category column has inconsistent capitalization and naming, such as Food vs. food and RainGear vs. rain-gear
5. The timestamps use different formats, like 2026-09-05T12:03:00


### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [11]:
removed = df.duplicated().sum()
clean = df.drop_duplicates().copy()
log("duplicates", removed, "drop exact duplicate rows")
clean


[duplicates] 1 (drop exact duplicate rows row(s))


,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [12]:
clean['price'] = clean['price'].str.replace('$', '', regex=False).str.strip().astype(float)

assert clean['price'].dtype == float

log("price", 0, "strip $/whitespace and convert price from text to float")

clean


[price] 0 (strip $/whitespace and convert price from text to float row(s))


,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,7.5,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12.0,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,24.0,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6.0,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,6.0,NaN
7,7,NaN,Merch,1.0,12.0,2026-09-05T14:00:00


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [13]:
#convert quantity to numeric
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

#count the problems before changing anything
missing = clean['qty'].isna().sum()
negative = (clean['qty'] < 0).sum()

# drop rows where quantity is missing
clean = clean.dropna(subset=['qty']).copy()

#log the two decisions separately
log("missing qty", missing, "drop rows with missing quantity")
log("negative qty", negative, "keep negative quantities as refunds")

clean

[missing qty] 1 (drop rows with missing quantity row(s))
[negative qty] 1 (keep negative quantities as refunds row(s))


,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,7.5,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
4,4,UVA T-Shirt,Apparel,2.0,24.0,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6.0,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,6.0,NaN
7,7,NaN,Merch,1.0,12.0,2026-09-05T14:00:00


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [14]:
print('before:', sorted(clean['category'].unique()))

before_count = clean['category'].nunique()

# lowercase, strip whitespace, and remove punctuation
clean['category'] = (
    clean['category']
    .str.lower()
    .str.strip()
    .str.replace(r'[^\w\s]', '', regex=True)
)

# map remaining variants that mean the same thing
CATEGORY_MAP = {
    'raingear': 'rain gear'
}

clean['category'] = clean['category'].replace(CATEGORY_MAP)

after_count = clean['category'].nunique()

print('after:', sorted(clean['category'].unique()))

log("categories", before_count - after_count,
    f"normalized categories from {before_count} to {after_count} distinct values")

clean

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after: ['apparel', 'food', 'merch', 'rain gear']
[categories] 2 (normalized categories from 6 to 4 distinct values row(s))


,order_id,item,category,qty,price,ts
0,1,Cheeseburger,food,2.0,7.5,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
4,4,UVA T-Shirt,apparel,2.0,24.0,2026-09-05 13:05
5,5,Rain Poncho,rain gear,-3.0,6.0,2026-09-05T13:20:00
6,6,rain poncho,rain gear,4.0,6.0,NaN
7,7,NaN,merch,1.0,12.0,2026-09-05T14:00:00


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [16]:
print("before:", sorted(clean['item'].dropna().unique()))

#normalize case and whitespace
clean['item'] = clean['item'].str.lower().str.strip()

#map different spellings of the same product
ITEM_MAP = {
    'cheese burger': 'cheeseburger'
}

clean['item'] = clean['item'].replace(ITEM_MAP)

print("after:", sorted(clean['item'].dropna().unique()))

#count and remove rows with missing item names
missing_item = clean['item'].isna().sum()
clean = clean.dropna(subset=['item']).copy()

log("missing item", missing_item, "drop rows with missing item name")

clean

before: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after: ['cheeseburger', 'rain poncho', 'uva t-shirt']
[missing item] 1 (drop rows with missing item name row(s))


,order_id,item,category,qty,price,ts
0,1,cheeseburger,food,2.0,7.5,2026-09-05T12:03:00
2,2,cheeseburger,food,1.0,7.5,09/05/2026 12:40
4,4,uva t-shirt,apparel,2.0,24.0,2026-09-05 13:05
5,5,rain poncho,rain gear,-3.0,6.0,2026-09-05T13:20:00
6,6,rain poncho,rain gear,4.0,6.0,NaN


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [18]:
# Convert timestamps to real datetimes
clean['ts'] = pd.to_datetime(
    clean['ts'],
    format='mixed',
    errors='coerce'
)

# Count timestamps that failed to parse
failed_ts = clean['ts'].isna().sum()

# Log the failures
log("timestamps", failed_ts, "coerce failed or missing timestamps to NaT")

# Create hour column
clean['hour'] = clean['ts'].dt.hour

clean

[timestamps] 1 (coerce failed or missing timestamps to NaT row(s))


,order_id,item,category,qty,price,ts,hour
0,1,cheeseburger,food,2.0,7.5,2026-09-05 12:03:00,12.0
2,2,cheeseburger,food,1.0,7.5,2026-09-05 12:40:00,12.0
4,4,uva t-shirt,apparel,2.0,24.0,2026-09-05 13:05:00,13.0
5,5,rain poncho,rain gear,-3.0,6.0,2026-09-05 13:20:00,13.0
6,6,rain poncho,rain gear,4.0,6.0,NaT,NaN


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [19]:
assert clean.duplicated().sum() == 0
assert clean['price'].dtype == float
assert clean['qty'].notna().all()
assert clean['item'].notna().all()
assert clean['category'].notna().all()
assert pd.api.types.is_datetime64_any_dtype(clean['ts'])

clean['revenue'] = clean['qty'] * clean['price']

print("Rows:", len(clean))
print("Units:", clean['qty'].sum())
print("Revenue:", clean['revenue'].sum())
print("Distinct categories:", clean['category'].nunique())

clean

Rows: 5
Units: 6.0
Revenue: 76.5
Distinct categories: 3


,order_id,item,category,qty,price,ts,hour,revenue
0,1,cheeseburger,food,2.0,7.5,2026-09-05 12:03:00,12.0,15.0
2,2,cheeseburger,food,1.0,7.5,2026-09-05 12:40:00,12.0,7.5
4,4,uva t-shirt,apparel,2.0,24.0,2026-09-05 13:05:00,13.0,48.0
5,5,rain poncho,rain gear,-3.0,6.0,2026-09-05 13:20:00,13.0,-18.0
6,6,rain poncho,rain gear,4.0,6.0,NaT,NaN,24.0


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [20]:
show_log()

,step,decision,rows
0,duplicates,1,drop exact duplicate rows
1,price,0,strip $/whitespace and convert price from text...
2,missing qty,1,drop rows with missing quantity
3,negative qty,1,keep negative quantities as refunds
4,categories,2,normalized categories from 6 to 4 distinct values
5,missing item,1,drop rows with missing item name
6,timestamps,1,coerce failed or missing timestamps to NaT
7,timestamps,1,coerce failed or missing timestamps to NaT


**The decision that mattered most:**
Keeping the negative quantity as a refund.


**Revenue with it:** $76.50   
**Revenue without it:** \$94.50

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [ ]:
# Checkpoint
rows_after = None            # TODO
revenue_after = None         # TODO
biggest_decision = 'TODO'    # TODO: which choice moved the number most
revenue_other_way = None     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)